# Assignment: When Chat Systems Break - A Realistic Sharding Simulation


---
## 📅 Day 10 (Final Analysis): Cross-Shard Query + System Evolution

In [ ]:
import random


class Message:
    def __init__(self, user_id, channel_id, content):
        self.user_id = user_id
        self.channel_id = channel_id
        self.content = content


class Shard:
    def __init__(self, shard_id):
        self.id = shard_id
        self.messages = []

    def store(self, message):
        self.messages.append(message)

    def count(self):
        return len(self.messages)


class ShardManager:
    def __init__(self, num_shards):
        self.shards = [Shard(i) for i in range(num_shards)]

    def send_message(self, message):
        # will be implemented by subclasses
        pass

    def print_stats(self):
        total = sum(s.count() for s in self.shards)
        print(f"{'Shard':<10} {'Messages':<15} {'% of Total':<10}")
        print('-' * 35)
        for shard in self.shards:
            pct = round((shard.count() / total * 100), 1) if total > 0 else 0
            print(f"Shard {shard.id:<5} {shard.count():<15} {pct}%")
        print(f"{'Total':<10} {total}")

import hashlib


class HashShardManager(ShardManager):
    def get_shard(self, key):
        h = int(hashlib.md5(str(key).encode()).hexdigest(), 16)
        return self.shards[h % len(self.shards)]

    def send_message(self, message):
        shard = self.get_shard(message.channel_id)
        shard.store(message)


In [ ]:
# =============================================
# DAY 10: Cross-Shard Query
# Task: Fetch last 10 messages of a channel
# Problem: Messages might be spread across shards
# =============================================

print("=== Cross-Shard Query: Fetch last 10 messages of a channel ===")
print()

# Setup: Use hash-based manager with some data
cross_mgr = HashShardManager(num_shards=3)

# Add messages with message IDs so we can sort them
msg_id = 0
for i in range(500):
    ch = random.randint(1, 10)
    msg = Message(user_id=random.randint(1, 100), channel_id=ch, content=f"msg-{msg_id}")
    msg.msg_id = msg_id   # add a simple message ID
    msg_id += 1
    cross_mgr.send_message(msg)


def fetch_last_n_from_channel(manager, channel_id, n=10):
    """
    Fetch last n messages of a channel.
    We have to check ALL shards because we don't know where data is.
    (In hash-based, channel maps to 1 shard. But in general, we check all.)
    """
    shards_checked = 0
    all_channel_msgs = []

    for shard in manager.shards:
        shards_checked += 1
        for msg in shard.messages:
            if msg.channel_id == channel_id:
                all_channel_msgs.append(msg)

    # Sort by msg_id to get order (latest = highest id)
    all_channel_msgs.sort(key=lambda m: m.msg_id, reverse=True)

    print(f"  Shards checked to find channel {channel_id}: {shards_checked}")
    print(f"  Total messages found for channel {channel_id}: {len(all_channel_msgs)}")
    print(f"  Last {n} messages (most recent first):")
    for m in all_channel_msgs[:n]:
        print(f"    [msg_id={m.msg_id}] User {m.user_id}: {m.content}")


fetch_last_n_from_channel(cross_mgr, channel_id=3, n=10)
print()
print("Observation: We had to scan ALL shards to find the messages.")
print("Cost: If 10 shards, we check 10 shards for every read query.")
print("This is why READ queries are harder than WRITE in sharded systems.")

In [ ]:
# =============================================
# DAY 10: System Evolution — 3 shards → 6 shards
# What breaks when shard count changes?
# =============================================

print("=== System Evolution: What happens when shards increase from 3 → 6? ===")
print()

import hashlib

def shard_for_channel(channel_id, num_shards):
    h = int(hashlib.md5(str(channel_id).encode()).hexdigest(), 16)
    return h % num_shards

print("Channel mapping BEFORE (3 shards):")
for ch in range(1, 11):
    print(f"  Channel {ch:>2} → Shard {shard_for_channel(ch, 3)}")

print()
print("Channel mapping AFTER adding more shards (6 shards):")
moved = 0
for ch in range(1, 11):
    old = shard_for_channel(ch, 3)
    new = shard_for_channel(ch, 6)
    flag = "  ← MOVED!" if old != new else ""
    if old != new:
        moved += 1
    print(f"  Channel {ch:>2} → was Shard {old}, now Shard {new}{flag}")

print()
print(f"Channels whose shard location CHANGED: {moved} out of 10")
print("Problem: Old data is still in the OLD shard. New messages go to NEW shard.")
print("Result: Queries find INCOMPLETE data — system is BROKEN!")
print("Solution (not implemented here): Consistent Hashing or manual data migration.")

In [ ]:
# =============================================
# DAY 10: Final Analysis — Answers to all questions
# =============================================

print("=" * 60)
print("FINAL ANALYSIS")
print("=" * 60)

print("""
Q1. Which shard failed first and why?
    In Channel-Based sharding:
    → Shard 0 (channel_id=3 % 3 = 0) failed first.
    → Because the viral cricket channel (channel 3) sent 80-95%
      of all traffic to shard 0, overloading it completely.
    In User-Based sharding:
    → Shard 1 (user_id=1 % 3 = 1) failed first.
    → Because the influencer (user 1) sent 5000 messages alone.

Q2. Which strategy looked good but failed under spike?
    → Channel-Based sharding looked logical at first.
    → Keeping channel messages together makes reads easier.
    → BUT during a viral event (1 channel dominates), it
      becomes a hotspot and fails completely.

Q3. What happens if shards increase from 3 → 10?
    → Hash functions use % num_shards.
    → When num_shards changes, almost all channel-to-shard
      mappings change.
    → Old data stays in old shards. New data goes to new shards.
    → Queries return INCOMPLETE results — system is broken.
    → This is why scaling up shards is very hard in practice.

Q4. What breaks when one shard goes down?
    → All messages that were going to that shard are LOST.
    → Any channel mapped to that shard becomes UNREADABLE.
    → Without replication (backup copies), data is gone forever.
    → This is why real systems like Discord use replication.
"""
)

print("=" * 60)
print("STRATEGY COMPARISON SUMMARY")
print("=" * 60)
print(f"{'Strategy':<20} {'Works Well When':<30} {'Fails When'}")
print("-" * 70)
print(f"{'User-Based':<20} {'Many equal-activity users':<30} {'One user is very active'}")
print(f"{'Channel-Based':<20} {'Many equal-traffic channels':<30} {'One channel goes viral'}")
print(f"{'Hash-Based':<20} {'Load spread across many keys':<30} {'One key dominates OR shard count changes'}")
print()
print("Conclusion: No single strategy is perfect.")
print("Real systems combine multiple approaches + replication + monitoring.")